In [ ]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu,pallas]==2.8.0",
                # 🩹 THE FIX: Unpinned numpy, pyarrow, and fsspec to allow Numpy 2.0+ compatibility
                "numpy", "pyarrow<16.0.0", "fsspec",
                "protobuf>=5.28.0",
                "datasets", "transformers", "huggingface_hub>=0.28.0", "wandb",
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

In [ ]:
%%writefile parallel_hardware_labeler.py
 
import io
import os
import sys
import time
import json
import site
import importlib
import warnings
import subprocess
import multiprocessing
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Union
 
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
 
from huggingface_hub import hf_hub_download, create_repo, HfApi
from huggingface_hub.utils import RepositoryNotFoundError, EntryNotFoundError
 
# Disable HF progress bars (noisy across 8 ranks)
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except ImportError:
    pass
 
# Ensure PJRT runtime gets selected, not XRT
for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
    os.environ.pop(key, None)
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
 
# Keep CPU threads tame during streaming/IO
os.environ["OMP_NUM_THREADS"] = "1"
pa.set_cpu_count(1)
pa.set_io_thread_count(1)
 
# PyTorch only on the accelerator
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
 
if 'site' in sys.modules:
    importlib.reload(site)
 
 
# Disable HF progress bars (noisy across 8 ranks)
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except ImportError:
    pass
 
# Ensure PJRT runtime gets selected, not XRT
for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
    os.environ.pop(key, None)
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
 
# Keep CPU threads tame during streaming/IO
os.environ["OMP_NUM_THREADS"] = "1"
pa.set_cpu_count(1)
pa.set_io_thread_count(1)
 
# PyTorch only on the accelerator
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
 
if 'site' in sys.modules:
    importlib.reload(site)
 
 
def get_secret(key_name):
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except Exception:
        pass
    return os.getenv(key_name)
 
 
def to_float(x):
    return x.item() if hasattr(x, 'item') else float(x)
 
 
# ==================================================
# HardwareConfig  (slimmed: we only need world size + dtype + device)
# ==================================================
@dataclass
class HardwareConfig:
    HARDWARE_PROFILES = {
        "v5e-8": {"ws": 8},
        "v5e-1": {"ws": 1},
        "v6e-1": {"ws": 1},
        "cpu":   {"ws": 1},
    }
 
    # Match the trainer's string so _parse_hardware behaves the same.
    hardware_string: str = "v5e-8 tpu"
    hf_token: str = ""
 
    # Filled in by _parse_hardware
    world_size: int = 1
    device_type: str = "cpu"
 
    # float32 keeps score fidelity for a small model; v5e has plenty of room.
    # Flip to "bf16" if you want raw speed and can tolerate rounding in scores.
    score_dtype: str = "float32"
 
 
# ==================================================
# LabelConfig  (replaces MLMDataConfig + CheckpointConfig)
# ==================================================
@dataclass
class LabelConfig:
    input_repo_id: str = "JamesResearch1216/ModernBERT-512-Combined-v5"
    output_repo_id: str = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v5"
    model_repo_id: str = "JamesResearch1216/ModernBERT-BT_Easiness_v5"
    tokenizer_name: str = "answerdotai/ModernBERT-base"

    subset_names: List[str] = field(default_factory=lambda: list(["seq_1024", "seq_2048", "seq_4096"]))
    bucket_thresholds: List[int] = field(default_factory=lambda: list([0, 72, 92]))
    bucket_pack_n: List[int] = field(default_factory=lambda: list([2, 4, 8]))

    train_indices: List[int] = field(default_factory=lambda: list(range(0, 113)))
    validation_work: List[Any] = field(default_factory=lambda: list([("validation", 0)]))
 
    batch_size: int = 256
    seq_len: int = 512
    drop_pack_remainder: bool = True
    input_subset_name: Optional[str] = "seq_512" 
 
    hf_token: str = ""
 
 
# ==================================================
# The fine-tuned difficulty/easiness model (copied from your inference nb)
# ==================================================
def build_regression_model(config):
    import torch.nn as nn
    from transformers import ModernBertModel
 
    class RegressionModel(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.modernBert = ModernBertModel(config)
            self.regression_head = nn.Sequential(
                nn.Linear(768, 256),
                nn.LayerNorm(256),
                nn.Tanh(),
                nn.Dropout(0.3),
                nn.Linear(256, 1),
            )
 
        def forward(self, input_ids, attention_mask, labels=None):
            outputs = self.modernBert(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden = outputs.last_hidden_state
            mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
            sum_embeddings = (last_hidden * mask).sum(1)
            sum_mask = mask.sum(1)
            mean_pooled = sum_embeddings / sum_mask
            logits = self.regression_head(mean_pooled)
            return {"logits": logits}
 
    return RegressionModel(config)
 
 
def load_regression_model(label_config: LabelConfig, hf_token: str):
    import torch
    from transformers import AutoConfig

    # (No download here — main process already fetched these before launch)
    config = AutoConfig.from_pretrained("./config.json")
    config._attn_implementation = "eager"
    if hasattr(config, "reference_compile"):
        config.reference_compile = False

    model = build_regression_model(config)
    state_dict = torch.load("./model.pt", map_location="cpu", weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    return model
 
 
# ==================================================
# HardwareDriver  (slimmed: parse + launch only)
# ==================================================
class HardwareDriver:
    def __init__(self, hw_config: HardwareConfig, label_config: LabelConfig):
        self.hw_config = hw_config
        self.label_config = label_config
        self._parse_hardware()
 
    def _parse_hardware(self):
        hardware_string = self.hw_config.hardware_string.lower().replace(" ", "")
 
        ws = None
        for key, prof in self.hw_config.HARDWARE_PROFILES.items():
            if key in hardware_string:
                ws = prof["ws"]
                break
        if ws is None:
            ws = 1
            warnings.warn("⚠️ hardware_string matched nothing. Falling back to single device.", UserWarning)
        self.hw_config.world_size = ws
 
        if "tpu" in hardware_string:
            self.hw_config.device_type = "tpu"
        elif any(x in hardware_string for x in ["gpu", "cuda", "a100", "p100", "h100", "t4", "l4"]):
            self.hw_config.device_type = "cuda"
        else:
            self.hw_config.device_type = "cpu"
 
    def launch(self, worker_fn):
        world_size = self.hw_config.world_size
        device = self.hw_config.device_type
 
        if world_size > 1 and device == "tpu":
            import torch_xla.distributed.xla_multiprocessing as xmp
            xmp.spawn(worker_fn, args=(self.hw_config, self.label_config), start_method='spawn')
        else:
            # Single device (TPU v5e-1, CPU sanity test, etc.)
            worker_fn(0, self.hw_config, self.label_config)
 
 
# ==================================================
# LabelStrategy  (replaces MLMDataStrategy; no shuffle, no shard)
# ==================================================
class LabelStrategy:
    def __init__(self, rank: int, world_size: int, config: LabelConfig, hf_token: str):
        self.rank = rank
        self.world_size = world_size
        self.config = config
        self.hf_token = hf_token
        self.local_dir = "./local_parquet_shards"
        os.makedirs(self.local_dir, exist_ok=True)
 
    # index -> (bucket_level, subset_name, pack_N)
    def assign_bucket(self, index: int):
        level = 0
        for k in range(len(self.config.bucket_thresholds)):
            if index >= self.config.bucket_thresholds[k]:
                level = k
        return level, self.config.subset_names[level], self.config.bucket_pack_n[level]
 
    def input_repo_path(self, split: str, index: int):
        return f"data/{self.config.input_subset_name}/{split}-{index:05d}.parquet"
 
    def output_repo_path(self, split: str, index: int):
        # Mirror the input layout so your existing reader/collator can consume it.
        _, subset, _ = self.assign_bucket(index)
        return f"data/{subset}/{split}-{index:05d}.parquet"
 
    def download_parquet(self, split: str, index: int):
        repo_path = self.input_repo_path(split, index)
        try:
            local_path = hf_hub_download(
                repo_id=self.config.input_repo_id,
                filename=repo_path,
                repo_type="dataset",
                token=self.hf_token,
                local_dir=self.local_dir,
                local_dir_use_symlinks=False,
            )
            return local_path
        except Exception as e:
            print(f"[rank ?] ❌ Failed to download {repo_path}: {e}")
            return ""
 
    def delete_parquet(self, local_path: str):
        try:
            if local_path and os.path.exists(local_path):
                os.remove(local_path)
        except Exception as e:
            print(f"⚠️ Failed to delete {local_path}: {e}")
 
    # Read input_ids in FILE ORDER as a contiguous [R, seq_len] int array.
    # No shuffle -> adjacency is preserved for packing.
    def read_input_ids(self, local_path: str) -> np.ndarray:
        table = pq.read_table(local_path, columns=["input_ids"])
        col = table.column("input_ids").combine_chunks()
        flat = col.values.to_numpy(zero_copy_only=False)
        total = flat.shape[0]
        if total % self.config.seq_len != 0:
            raise ValueError(
                f"{local_path}: flat token count {total} is not a multiple of "
                f"{self.config.seq_len}; rows are not uniform 512."
            )
        ids = flat.reshape(-1, self.config.seq_len).astype(np.int32, copy=False)
        return ids
 
 
# Resume support: which output files already exist in the output repo.
def get_done_set(hf_token: str, output_repo_id: str):
    api = HfApi(token=hf_token)
    try:
        return set(api.list_repo_files(repo_id=output_repo_id, repo_type="dataset"))
    except Exception as e:
        print(f"(could not list output repo, assuming empty: {e})")
        return set()
 
 
# ==================================================
# Scoring + packing
# ==================================================
def score_rows(model, device, ids_np: np.ndarray, batch_size: int, is_tpu: bool, score_dtype, rank: int) -> np.ndarray:
    """Return one easiness score per row, in input order. Pads final batch and slices."""
    import torch

    R = ids_np.shape[0]
    seq_len = ids_np.shape[1]
    scores = np.empty(R, dtype=np.float32)

    if is_tpu:
        import torch_xla.core.xla_model as xm

    total_batches = (R + batch_size - 1) // batch_size

    with torch.no_grad():
        for i, s in enumerate(range(0, R, batch_size)):
            # Print an update every 10 batches, or on the very last batch
            if i % 100 == 0 or i == total_batches - 1:
                print(f"[rank {rank}] ⚙️ Scoring batch {i+1}/{total_batches} ({(i+1)/total_batches*100:.1f}%)", flush=True)

            e = min(s + batch_size, R)
            n = e - s
            chunk = ids_np[s:e]

            # Pad the final ragged batch up to batch_size so XLA keeps ONE graph.
            if n < batch_size:
                pad = np.zeros((batch_size - n, seq_len), dtype=np.int32)
                chunk = np.concatenate([chunk, pad], axis=0)

            input_ids = torch.from_numpy(chunk).long().to(device)
            # No padding inside rows (fully packed 512) -> mask is all ones.
            attention_mask = torch.ones_like(input_ids)

            out = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = out["logits"].squeeze(-1).float()  # [batch_size]

            if is_tpu:
                xm.mark_step()

            batch_scores = logits.cpu().numpy()
            batch_scores = np.clip(batch_scores, 0.0, 1.0)
            scores[s:e] = batch_scores[:n]  # discard padded rows

    return scores
 
 
def pack_rows(ids_np: np.ndarray, scores: np.ndarray, pack_n: int, drop_remainder: bool):
    """Concatenate N adjacent rows; average their scores. Returns (list[list[int]], list[float])."""
    R = ids_np.shape[0]
    n_full = R // pack_n
    if not drop_remainder and (R % pack_n) != 0:
        # If you ever want to keep the tail, you'd pad here. Default drops it.
        pass
 
    packed_ids = []
    packed_scores = []
    for g in range(n_full):
        lo = g * pack_n
        hi = lo + pack_n
        seq = ids_np[lo:hi].reshape(-1).tolist()       # N * 512 ints
        packed_ids.append(seq)
        packed_scores.append(float(scores[lo:hi].mean()))
    return packed_ids, packed_scores
 
 
def write_output_parquet(packed_ids, packed_scores, local_filename):
    # Use datasets.Dataset to match the exact on-disk format your pipeline produced.
    import datasets
    ds = datasets.Dataset.from_dict({
        "input_ids": packed_ids,
        "easiness_score": packed_scores,
    })
    ds.to_parquet(local_filename)
 
 
def ping_sidecar(local_filename, repo_path, index):
    request = {"local_file": local_filename, "repo_path": repo_path, "index": index}
    tmp = f"UPLOAD_REQUEST_{index}.json.tmp"
    final = f"UPLOAD_REQUEST_{index}.json"
    with open(tmp, "w") as f:
        json.dump(request, f)
    os.rename(tmp, final)

# ==================================================
# label_worker  (replaces train_worker)
# ==================================================
def label_worker(rank, hw_config: HardwareConfig, label_config: LabelConfig):
    import threading
    import traceback
    import time
    import os

    # ---- heartbeat (same mechanism as the trainer) ----
    heartbeat_stop = threading.Event()

    def heartbeat_report():
        path = f"/tmp/heartbeat_rank_{rank}.txt"
        while not heartbeat_stop.wait(timeout=10):
            try:
                with open(path, "w") as f:
                    f.write(str(time.time()))
            except Exception:
                pass

    threading.Thread(target=heartbeat_report, daemon=True).start()

    if hw_config.hf_token:
        os.environ["HF_TOKEN"] = hw_config.hf_token

    try:
        import torch

        is_tpu = (hw_config.device_type == "tpu")
        if is_tpu:
            import torch_xla.core.xla_model as xm
            import torch_xla.runtime as xr
            device = xm.xla_device()
            real_ws = xr.world_size()
            if hw_config.world_size != real_ws:
                if rank == 0:
                    print(f"⚠️ Adjusting world size {hw_config.world_size} -> {real_ws}", flush=True)
                hw_config.world_size = real_ws
        elif hw_config.device_type == "cuda":
            torch.cuda.set_device(rank)
            device = torch.device(f"cuda:{rank}")
        else:
            device = torch.device("cpu")

        score_dtype = torch.float32 if hw_config.score_dtype == "float32" else torch.bfloat16

        if rank == 0:
            print(f"Rank 0 online on {device}. Loading model... (XLA compilation on first batch will take a few minutes!)", flush=True)

        model = load_regression_model(label_config, hw_config.hf_token).to(device)
        if score_dtype == torch.bfloat16:
            model = model.to(torch.bfloat16)
        model.eval()

        strat = LabelStrategy(rank, hw_config.world_size, label_config, hw_config.hf_token)

        # Build this rank's work list: parquet i -> rank (i % world_size)
        work = []
        
        # 1. Standard index-based routing for Training Data
        for idx in label_config.train_indices:
            if idx % hw_config.world_size == rank:
                work.append(("train", idx, None)) # None = Use standard bucket logic

        # 2. Explicit routing for Validation Data (Process index 0 three times)
        validation_overrides = [
            ("seq_1024", 2),
            ("seq_2048", 4),
            ("seq_4096", 8)
        ]
        for task_id, override in enumerate(validation_overrides):
            # Distribute the 3 validation tasks across available TPU ranks
            if task_id % hw_config.world_size == rank:
                work.append(("validation", 0, override))

        done = get_done_set(hw_config.hf_token, label_config.output_repo_id)

        if rank == 0:
            print(f"Rank {rank}: {len(work)} parquet(s) assigned. {len(done)} output file(s) already present.", flush=True)

        for (split, idx, override) in work:
            
            # Use overrides for Validation, otherwise use standard index logic for Train
            if override is None:
                level, subset, pack_n = strat.assign_bucket(idx)
                out_path = strat.output_repo_path(split, idx)
            else:
                subset, pack_n = override
                out_path = f"data/{subset}/{split}-{idx:05d}.parquet"

            if out_path in done:
                print(f"[rank {rank}] ⏭️  {out_path} already labeled. Skipping.", flush=True)
                continue

            t0 = time.time()

            print(f"[rank {rank}] 📥 Downloading {split}-{idx:05d}...", flush=True)
            local_in = strat.download_parquet(split, idx)
            if not local_in:
                print(f"[rank {rank}] ⚠️ Skipping {split}-{idx:05d} (download failed).", flush=True)
                continue

            print(f"[rank {rank}] 📦 Download complete. Reading into memory...", flush=True)
            ids_np = strat.read_input_ids(local_in)
            R = ids_np.shape[0]

            print(f"[rank {rank}] 🧠 Starting inference on {R} rows...", flush=True)
            scores = score_rows(
                model, device, ids_np,
                batch_size=label_config.batch_size,
                is_tpu=is_tpu, score_dtype=score_dtype,
                rank=rank
            )

            print(f"[rank {rank}] 🧳 Packing {R} rows into target buckets...", flush=True)
            packed_ids, packed_scores = pack_rows(
                ids_np, scores, pack_n, label_config.drop_pack_remainder
            )

            local_out = f"out-{split}-{idx:05d}.parquet"
            write_output_parquet(packed_ids, packed_scores, local_out)

            ping_sidecar(local_out, out_path, f"{split}_{idx:05d}")
            strat.delete_parquet(local_in)

            if is_tpu:
                xm.mark_step()

            dt = time.time() - t0
            print(f"[rank {rank}] ✅ {split}-{idx:05d} ({subset}, x{pack_n}): "
                  f"{R} rows -> {len(packed_ids)} packed in {dt:.0f}s "
                  f"(score range {scores.min():.3f}..{scores.max():.3f})", flush=True)

        print(f"[rank {rank}] 🏁 All assigned parquets done.", flush=True)

    except Exception:
        import traceback as _tb
        err = _tb.format_exc()
        print(f"\n❌ FATAL WORKER ERROR ON RANK {rank}:\n{err}", flush=True)
        return
    finally:
        heartbeat_stop.set()
        try:
            os.remove(f"/tmp/heartbeat_rank_{rank}.txt")
        except FileNotFoundError:
            pass
        except Exception as e:
            print(f"[rank {rank}] ⚠️ Could not clean up heartbeat file: {e}", flush=True)

 
 
# ==================================================
# Sidecar uploader  (adapted from the trainer's sidecar)
# Uploads finished packed parquets to the OUTPUT dataset repo.
# ==================================================
def sidecar_uploader_loop(hf_token, output_repo_id):
    import signal
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    signal.signal(signal.SIGTERM, signal.SIG_IGN)

    api = HfApi(token=hf_token)
    idle_ticks = 0

    while True:
        requests = [f for f in os.listdir(".")
                    if f.startswith("UPLOAD_REQUEST_") and f.endswith(".json")]
        if requests:
            idle_ticks = 0
            req_file = sorted(requests)[0]
            try:
                with open(req_file) as f:
                    req = json.load(f)
                local_file = req["local_file"]
                repo_path = req["repo_path"]

                if os.path.exists(local_file):
                    print(f"⏳ Uploading {local_file} -> {output_repo_id}/{repo_path}", flush=True)
                    api.upload_file(
                        path_or_fileobj=local_file,
                        path_in_repo=repo_path,
                        repo_id=output_repo_id,
                        repo_type="dataset",
                    )
                    os.remove(local_file)
                    print(f"✅ Uploaded {repo_path}", flush=True)
                else:
                    print(f"❌ {req_file} pinged but {local_file} missing.", flush=True)
                os.remove(req_file)
            except Exception as e:
                print(f"❌ Upload failed for {req_file}: {e}. Retrying in 5s...", flush=True)
        else:
            idle_ticks += 1
            if idle_ticks % 12 == 0:  # Every ~60 seconds
                print("🛸 Sidecar standing by, waiting for workers to finish a parquet...", flush=True)
        time.sleep(5)
 
 
def salvage_pending(hf_token, output_repo_id, max_total_seconds=300):
    """Drain any leftover UPLOAD_REQUEST files on shutdown."""
    api = HfApi(token=hf_token)
    deadline = time.time() + max_total_seconds
    pending = sorted(f for f in os.listdir(".")
                     if f.startswith("UPLOAD_REQUEST_") and f.endswith(".json"))
    if not pending:
        return
    print(f"💾 Salvaging {len(pending)} pending upload(s)...")
    for req_file in pending:
        if time.time() > deadline:
            print("⚠️ Salvage budget exhausted.")
            break
        try:
            with open(req_file) as f:
                req = json.load(f)
            local_file = req["local_file"]
            repo_path = req["repo_path"]
            if not os.path.exists(local_file):
                os.remove(req_file)
                continue
            api.upload_file(path_or_fileobj=local_file, path_in_repo=repo_path,
                            repo_id=output_repo_id, repo_type="dataset")
            os.remove(local_file)
            os.remove(req_file)
            print(f"✅ Salvaged {repo_path}")
        except Exception as e:
            print(f"❌ Failed to salvage {req_file}: {e}")
 
 
# ==================================================
# Main: spawn sidecar + watchdogs, launch workers, clean up
# ==================================================
if __name__ == "__main__":
    import signal
    import threading
    hf_token = get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
 
    shutdown_event = threading.Event()
    shutdown_count = {"n": 0}
 
    def graceful_shutdown(signum, frame):
        shutdown_count["n"] += 1
        if shutdown_count["n"] >= 2:
            print(f"2nd signal {signum}. Hard exit.")
            try:
                with open("/tmp/USER_STOPPED_LABELING", "w") as f:
                    f.write("hard")
            except Exception:
                pass
            os._exit(1)
        else:
            print(f"Signal {signum}. Graceful shutdown... (again to hard-exit)")
            try:
                with open("/tmp/USER_STOPPED_LABELING", "w") as f:
                    f.write("graceful")
            except Exception:
                pass
            shutdown_event.set()
            raise KeyboardInterrupt()
 
    signal.signal(signal.SIGTERM, graceful_shutdown)
    signal.signal(signal.SIGINT, graceful_shutdown)
 
    for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
        os.environ.pop(key, None)
    os.environ["PJRT_DEVICE"] = "TPU"
 
    hf_token = get_secret("HF_TOKEN")
 
    HW_CFG = HardwareConfig(hf_token=hf_token)
    LBL_CFG = LabelConfig(hf_token=hf_token)
 
    # Create the output repo once, up front (main process).
    try:
        create_repo(repo_id=LBL_CFG.output_repo_id, token=hf_token,
                    repo_type="dataset", private=False, exist_ok=True)
        print(f"🏗️  Output repo ready: {LBL_CFG.output_repo_id}")
    except Exception as e:
        print(f"⚠️ Could not pre-create output repo: {e}")
 
    ctx = multiprocessing.get_context('spawn')
    sidecar_holder = {"proc": None}
 
    def spawn_sidecar():
        p = ctx.Process(target=sidecar_uploader_loop,
                        args=(LBL_CFG.hf_token, LBL_CFG.output_repo_id),
                        daemon=False)
        p.start()
        return p
 
    sidecar_holder["proc"] = spawn_sidecar()
 
    def sidecar_watchdog():
        crash_count = 0
        MAX_CRASHES = 5
        while not shutdown_event.wait(timeout=30):
            proc = sidecar_holder["proc"]
            if not proc.is_alive():
                proc.join(timeout=5)
                if crash_count >= MAX_CRASHES:
                    print("Sidecar crashed too many times. No more respawns.")
                    return
                crash_count += 1
                print(f"⚠️ Sidecar died (exit={proc.exitcode}, #{crash_count}). Respawning...")
                sidecar_holder["proc"] = spawn_sidecar()
 
    threading.Thread(target=sidecar_watchdog, daemon=True).start()
 
    def label_watchdog():
        STALE_WARN = 120
        STALE_KILL = 600
        warned = set()
        while not shutdown_event.wait(timeout=30):
            now = time.time()
            for r in range(HW_CFG.world_size):
                path = f"/tmp/heartbeat_rank_{r}.txt"   # NOTE: underscore (trainer had a typo here)
                if not os.path.exists(path):
                    continue
                try:
                    with open(path) as f:
                        last = float(f.read().strip())
                except Exception:
                    continue
                age = now - last
                if STALE_KILL is not None and age > STALE_KILL:
                    print(f"💀 Rank {r} heartbeat {age:.0f}s old. Hard-killing.")
                    shutdown_event.set()
                    os._exit(2)
                if age > STALE_WARN and r not in warned:
                    print(f"⚠️ Rank {r} heartbeat {age:.0f}s old. Possible hang.")
                    warned.add(r)
                elif age <= STALE_WARN and r in warned:
                    print(f"✅ Rank {r} heartbeat recovered.")
                    warned.discard(r)
 
    threading.Thread(target=label_watchdog, daemon=True).start()

    # Pre-download the model weights ONCE in the main process.
    # All 8 worker processes will then load from the same local files.
    print("⏳ Pre-fetching model weights...")
    hf_hub_download(repo_id=LBL_CFG.model_repo_id, filename="model.pt",
                    repo_type="model", token=hf_token, local_dir=".", local_dir_use_symlinks=False)
    hf_hub_download(repo_id=LBL_CFG.model_repo_id, filename="config.json",
                    repo_type="model", token=hf_token, local_dir=".", local_dir_use_symlinks=False)
    print("✅ Model weights on disk.")
    
    driver = HardwareDriver(HW_CFG, LBL_CFG)
 
    try:
        driver.launch(label_worker)
    except KeyboardInterrupt:
        print("⚠️ Labeling interrupted by signal.")
    except Exception as e:
        print(f"💀 Launch error: {e}")
        import traceback
        traceback.print_exc()
    finally:
        print("🔪 Termination sequence...")
        shutdown_event.set()
 
        # Reap orphaned worker processes (not the sidecar)
        try:
            orphans = [p for p in multiprocessing.active_children()
                       if p is not sidecar_holder["proc"]]
            for o in orphans:
                try:
                    o.terminate()
                except Exception:
                    pass
            deadline = time.time() + 20
            for o in orphans:
                o.join(timeout=max(0, deadline - time.time()))
            for o in orphans:
                if o.is_alive():
                    try:
                        o.kill()
                    except Exception:
                        pass
                    o.join(timeout=5)
        except Exception as e:
            print(f"⚠️ Error reaping orphans: {e}")
 
        # Let the sidecar drain outstanding uploads
        elapsed = 0
        MAX_DRAIN = 600
        while elapsed < MAX_DRAIN:
            still = any(f.startswith("UPLOAD_REQUEST_") and f.endswith(".json")
                        for f in os.listdir("."))
            if not still:
                break
            if not sidecar_holder["proc"].is_alive():
                print("Sidecar died with uploads pending; will salvage.")
                break
            if elapsed % 60 == 0:
                print(f"⏳ Sidecar draining uploads... {elapsed // 60} min")
            time.sleep(5)
            elapsed += 5
 
        salvage_pending(hf_token, LBL_CFG.output_repo_id, max_total_seconds=300)
 
        try:
            sidecar_holder["proc"].kill()
            sidecar_holder["proc"].join(timeout=10)
        except Exception:
            pass
 
        # Clear the libtpu lockfile so "stop and re-run" works again
        if HW_CFG.device_type == "tpu":
            for lockfile in ["/tmp/libtpu_lockfile"]:
                if os.path.exists(lockfile):
                    try:
                        os.remove(lockfile)
                        print(f"🧹 Removed stale {lockfile}")
                    except Exception as e:
                        print(f"⚠️ Could not remove {lockfile}: {e}")
 
        for i in range(HW_CFG.world_size):
            path = f"/tmp/heartbeat_rank_{i}.txt"
            if os.path.exists(path):
                try:
                    os.remove(path)
                except Exception:
                    pass
 
        print("💅 Labeling shutdown ✨COMPLETE✨")

In [ ]:
# same retry/backoff launcher you had, but:
# import subprocess
# proc = subprocess.Popen(["python", "-u", "parallel_hardware_labeler.py"], start_new_session=True)
# and change USER_STOP_MARKER to "/tmp/USER_STOPPED_LABELING"

!python -u parallel_hardware_labeler.py